# Player co-play graph on the raw dumps

Who played whom across **all** games in `data/raw/` (no sampling, no rapid filter, no Elo-difference filter). Numbers only.

1. Scan the three `.pgn.zst` files for headers only (byte scan, no PGN parsing) and cache them in `data/processed/raw_games_headers.parquet`.
2. Build the graph: node = player, edge = at least one game together, edge weight = number of games.
3. Print stats for all games and per speed.

`RAPID_ONLY = True` restricts the main graph to rapid games. The 2017 dumps have no "Rapid" event label (rapid games are tagged Classical), so rapid is derived from `TimeControl` with the sampler's rule: base + 40 * increment in [480, 1499] s. No other sampling filter (bots, Elo difference, length) is applied.

In [1]:
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path
import io
import time

import numpy as np
import pandas as pd
import polars as pl
import zstandard as zstd
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from IPython.display import display

ROOT = Path.cwd().parent
RAW = sorted((ROOT / "data" / "raw").glob("*.pgn.zst"))
CACHE = ROOT / "data" / "processed" / "raw_games_headers.parquet"
RAPID_ONLY = True   # False = all games
BIN_EDGES = [800, 1000, 1200, 1400, 1600, 1800, 2000, 2200]

pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.3f}".format)
print([p.name for p in RAW])

['lichess_db_standard_rated_2017-04.pgn.zst', 'lichess_db_standard_rated_2017-05.pgn.zst', 'lichess_db_standard_rated_2017-06.pgn.zst']


## 1. Header scan (cached)

In [2]:
PREFIXES = {  # header line prefix -> column
    b"[Event \"": "event", b"[White \"": "white", b"[Black \"": "black",
    b"[TimeControl \"": "time_control", b"[WhiteElo \"": "white_elo", b"[BlackElo \"": "black_elo",
}
COLS = list(PREFIXES.values())
CHUNK = 1_000_000


def scan_file(path):
    """Headers only. A game starts at its [Event line; flush the previous one."""
    frames, cols, cur = [], {c: [] for c in COLS}, None

    def flush():
        for c in COLS:
            cols[c].append(cur.get(c))

    with open(path, "rb") as fh:
        buf = io.BufferedReader(zstd.ZstdDecompressor().stream_reader(fh), buffer_size=1 << 22)
        for line in buf:
            if not line.startswith(b"["):
                continue
            for prefix, col in PREFIXES.items():
                if line.startswith(prefix):
                    if col == "event":
                        if cur is not None:
                            flush()
                            if len(cols["event"]) >= CHUNK:
                                frames.append(pl.DataFrame(cols))
                                cols = {c: [] for c in COLS}
                        cur = {}
                    if cur is not None:
                        cur[col] = line[len(prefix):line.rindex(b'"')].decode("utf-8", "replace")
                    break
    if cur is not None:
        flush()
    frames.append(pl.DataFrame(cols))
    return pl.concat(frames)


if CACHE.exists():
    raw = pl.read_parquet(CACHE)
    print("loaded cache")
else:
    t0 = time.time()
    with ProcessPoolExecutor(max_workers=len(RAW)) as ex:
        raw = pl.concat(list(ex.map(scan_file, RAW)))
    raw = raw.with_columns(
        pl.col("event").str.split(" ").list.get(1).alias("speed"),
        pl.col("white").str.to_lowercase(),
        pl.col("black").str.to_lowercase(),
        pl.col("white_elo").cast(pl.Int32, strict=False),
        pl.col("black_elo").cast(pl.Int32, strict=False),
    ).drop("event")
    tc = raw["time_control"].str.extract_groups(r"^(\d+)\+(\d+)$")
    est = tc.struct[0].cast(pl.Int64, strict=False) + 40 * tc.struct[1].cast(pl.Int64, strict=False)
    raw = raw.with_columns(est.is_between(480, 1499).fill_null(False).alias("rapid"))
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    raw.write_parquet(CACHE)
    print(f"scanned in {time.time() - t0:.0f}s")

print(f"games: {raw.height:,}   (project_report.md says 34,555,025)")
display(raw.head(3).to_pandas())
display(raw.group_by("speed", "rapid").len().sort("len", descending=True).to_pandas())
display(raw["speed"].value_counts().sort("count", descending=True).to_pandas())
print("null elo:", raw["white_elo"].null_count(), raw["black_elo"].null_count())

scanned in 99s
games: 34,555,025   (project_report.md says 34,555,025)


,white,black,time_control,white_elo,black_elo,speed,rapid
0,calvinmaster,dislikechess,180+0,2186,1907,Blitz,False
1,gregster101,flavietta,120+1,1385,1339,Bullet,False
2,napen,volcoom,180+0,1905,1836,Blitz,False


,speed,rapid,len
0,Blitz,False,16273513
1,Bullet,False,9072053
2,Classical,True,7637833
3,UltraBullet,False,837531
4,Classical,False,613461
5,Correspondence,False,120634


,speed,count
0,Blitz,16273513
1,Bullet,9072053
2,Classical,8251294
3,UltraBullet,837531
4,Correspondence,120634


null elo: 0 0


## 2. Graph statistics

In [3]:
def graph_stats(white, black):
    """white, black: polars string Series (same length). Returns a stats dict and per-node arrays."""
    n_games_in = len(white)
    allp = pl.concat([white, black]).alias("p")
    codes = allp.cast(pl.Categorical).to_physical().to_numpy().astype(np.int64)
    names = pl.DataFrame({"p": allp, "c": codes}).unique("c")  # code -> username
    s, d = codes[:n_games_in], codes[n_games_in:]
    keep = s != d                                  # self-play
    s, d = s[keep], d[keep]
    m = len(s)

    ids, inv = np.unique(np.concatenate([s, d]), return_inverse=True)
    n = len(ids)
    s, d = inv[:m], inv[m:]
    key, weight = np.unique(np.minimum(s, d) * n + np.maximum(s, d), return_counts=True)
    a, b = key // n, key % n
    degree = np.bincount(np.concatenate([a, b]), minlength=n)
    games_pp = np.bincount(inv, minlength=n)

    adj = coo_matrix((np.ones(len(key), np.int8), (a, b)), shape=(n, n))
    n_comp, comp = connected_components(adj, directed=False)
    csize = np.bincount(comp)
    giant = csize.argmax()
    giant_games = int((comp[s] == giant).sum())

    stats = {
        "games": m, "players": n, "pairings": len(key),
        "mean degree": degree.mean(), "median degree": np.median(degree),
        "p90 degree": np.percentile(degree, 90), "p99 degree": np.percentile(degree, 99),
        "max degree": degree.max(),
        "pairings met >1 time": (weight > 1).mean(),
        "components": n_comp,
        "giant: players": csize[giant], "giant: % players": csize[giant] / n,
        "giant: games": giant_games, "giant: % games": giant_games / m,
        "components < 10 players": int((csize < 10).sum()),
        "players with 1 game": (games_pp == 1).mean(),
        "players with >=10 games": (games_pp >= 10).mean(),
        "median games/player": np.median(games_pp), "max games/player": games_pp.max(),
    }
    extra = dict(ids=ids, names=names, degree=degree, games_pp=games_pp, csize=csize)
    return stats, extra


df = raw.filter(pl.col("rapid")) if RAPID_ONLY else raw
t0 = time.time()
stats, extra = graph_stats(df["white"], df["black"])
print(f"rapid_only={RAPID_ONLY}  built in {time.time() - t0:.0f}s")
display(pd.Series(stats, name="rapid" if RAPID_ONLY else "all").to_frame())

rapid_only=True  built in 1s


,rapid
games,"7,637,833.000"
players,"208,226.000"
pairings,"6,184,564.000"
mean degree,59.402
median degree,9.000
p90 degree,180.000
p99 degree,615.000
max degree,"3,267.000"
pairings met >1 time,0.150
components,"1,025.000"


In [4]:
# component sizes
csize = extra["csize"]
edges = [1, 2, 3, 10, 100, 1000, 10**6, csize.max() + 1]
labels = ["1", "2", "3-9", "10-99", "100-999", "1000-999999", "giant"]
cut = pd.cut(csize, bins=[0, 1, 2, 9, 99, 999, csize.max() - 1, csize.max()], labels=labels)
comp_tab = pd.DataFrame({"size": csize, "bucket": cut}).groupby("bucket", observed=True)["size"].agg(components="size", players="sum")
display(comp_tab)

,components,players
bucket,,
2,919,1838
3-9,104,362
10-99,1,12
giant,1,206014


In [5]:
# top players by games and by distinct opponents
lut = dict(zip(extra["names"]["c"].to_list(), extra["names"]["p"].to_list()))
name_of = lambda i: lut[int(extra["ids"][i])]
top = lambda arr: [(name_of(i), int(arr[i])) for i in np.argsort(arr)[::-1][:20]]
display(pd.DataFrame({"by games": top(extra["games_pp"]), "by distinct opponents": top(extra["degree"])}))

,by games,by distinct opponents
0,"(albi71, 4775)","(jeanaymar, 3267)"
1,"(jeanaymar, 4677)","(keplee, 2911)"
2,"(keplee, 4472)","(albi71, 2637)"
3,"(tutubelezza, 3947)","(tutubelezza, 2483)"
4,"(yory21, 3844)","(inmal, 2314)"
5,"(esteban1961, 3768)","(yory21, 2298)"
6,"(vahidrabiee, 3463)","(vox767, 2294)"
7,"(gen3lsx, 3390)","(jagodin70, 2211)"
8,"(putra_pandawa, 3141)","(georgezig, 2079)"
9,"(hartonorahman, 3136)","(vahidrabiee, 2076)"


## 3. Elo-bin pairing matrix

Per-game Elo of each side (a player's mean would mix ratings across speeds). Symmetrised: cell (i, j) = games between a player in bin i and one in bin j.

In [6]:
BIN_NAMES = (["<800"] + [f"[{lo}, {hi})" for lo, hi in zip(BIN_EDGES[:-1], BIN_EDGES[1:])] + [">=2200"])
e = df.drop_nulls(["white_elo", "black_elo"])
bw = np.digitize(e["white_elo"].to_numpy(), BIN_EDGES)
bb = np.digitize(e["black_elo"].to_numpy(), BIN_EDGES)
k = len(BIN_NAMES)
M = np.bincount(bw * k + bb, minlength=k * k).reshape(k, k)
M = M + M.T - np.diag(np.diag(M))
mat = pd.DataFrame(M, index=BIN_NAMES, columns=BIN_NAMES)
display(mat)
display((mat.div(mat.sum(axis=1), axis=0) * 100).round(1))
print(f"games with both elos: {len(e):,}")
print(f"same bin: {(bw == bb).mean():.1%}   within one bin: {(np.abs(bw - bb) <= 1).mean():.1%}")
print("games per bin (player slots):"); display(mat.sum(axis=1).rename("slots").to_frame())

,<800,"[800, 1000)","[1000, 1200)","[1200, 1400)","[1400, 1600)","[1600, 1800)","[1800, 2000)","[2000, 2200)",>=2200
<800,38,1799,1787,266,156,79,25,5,1
"[800, 1000)",1799,23517,77802,16528,5566,2405,758,150,48
"[1000, 1200)",1787,77802,238219,272323,63119,23317,6443,1169,184
"[1200, 1400)",266,16528,272323,741316,612488,140106,35446,5724,899
"[1400, 1600)",156,5566,63119,612488,1321493,858023,145876,24526,3592
"[1600, 1800)",79,2405,23317,140106,858023,1250109,650431,79549,10032
"[1800, 2000)",25,758,6443,35446,145876,650431,617569,223921,23679
"[2000, 2200)",5,150,1169,5724,24526,79549,223921,109736,36517
>=2200,1,48,184,899,3592,10032,23679,36517,11097


,<800,"[800, 1000)","[1000, 1200)","[1200, 1400)","[1400, 1600)","[1600, 1800)","[1800, 2000)","[2000, 2200)",>=2200
<800,0.900,43.300,43.000,6.400,3.800,1.900,0.600,0.100,0.000
"[800, 1000)",1.400,18.300,60.500,12.900,4.300,1.900,0.600,0.100,0.000
"[1000, 1200)",0.300,11.400,34.800,39.800,9.200,3.400,0.900,0.200,0.000
"[1200, 1400)",0.000,0.900,14.900,40.600,33.600,7.700,1.900,0.300,0.000
"[1400, 1600)",0.000,0.200,2.100,20.200,43.500,28.300,4.800,0.800,0.100
"[1600, 1800)",0.000,0.100,0.800,4.600,28.500,41.500,21.600,2.600,0.300
"[1800, 2000)",0.000,0.000,0.400,2.100,8.600,38.200,36.200,13.100,1.400
"[2000, 2200)",0.000,0.000,0.200,1.200,5.100,16.500,46.500,22.800,7.600
>=2200,0.000,0.100,0.200,1.000,4.200,11.700,27.500,42.400,12.900


games with both elos: 7,637,833
same bin: 56.5%   within one bin: 92.3%
games per bin (player slots):


,slots
<800,4156
"[800, 1000)",128573
"[1000, 1200)",684363
"[1200, 1400)",1825096
"[1400, 1600)",3034839
"[1600, 1800)",3014051
"[1800, 2000)",1704148
"[2000, 2200)",481297
>=2200,86049


## 4. Per speed

In [7]:
rows = {}
for sp in raw["speed"].unique().sort().to_list():
    sub = raw.filter(pl.col("speed") == sp)
    rows[sp], _ = graph_stats(sub["white"], sub["black"])
    print("done", sp)
per_speed = pd.DataFrame(rows)
rp = raw.filter(pl.col("rapid"))
per_speed["rapid (by time control)"] = pd.Series(graph_stats(rp["white"], rp["black"])[0])
per_speed["all"] = pd.Series(graph_stats(raw["white"], raw["black"])[0])
display(per_speed)

done Blitz


done Bullet


done Classical
done Correspondence
done UltraBullet


,Blitz,Bullet,Classical,Correspondence,UltraBullet,rapid (by time control),all
games,"16,273,513.000","9,072,053.000","8,251,294.000","120,634.000","837,531.000","7,637,833.000","34,555,025.000"
players,"229,058.000","126,343.000","226,915.000","30,893.000","23,174.000","208,226.000","361,211.000"
pairings,"12,009,845.000","4,703,652.000","6,689,051.000","63,516.000","279,000.000","6,184,564.000","23,651,168.000"
mean degree,104.863,74.458,58.956,4.112,24.079,59.402,130.955
median degree,16.000,5.000,9.000,1.000,3.000,9.000,22.000
p90 degree,326.000,200.000,178.000,8.000,48.000,180.000,399.000
p99 degree,"1,009.430","1,070.000",608.000,53.000,397.270,615.000,"1,209.900"
max degree,"3,334.000","4,966.000","3,267.000",690.000,"2,779.000","3,267.000","6,050.000"
pairings met >1 time,0.194,0.318,0.149,0.227,0.453,0.150,0.210
components,457.000,367.000,"1,223.000","4,482.000",123.000,"1,025.000","1,497.000"
